## Intro

Tinygrad just dropped this cryptic tweet

![tinygrad-tweet](tinygrad_tweet.png)

George basically name-dropped all the secret compiler sauce that makes tinygrad work -- Halide, E-graphs, ILP, ThunderKittens -- but didn't explain any of it. This is the tinygrad roadmap. These are the techniques that will hopefully let tinygrad beat PyTorch, write the fastest kernels, and deliver on their $2M [AMD contract](https://x.com/__tinygrad__/status/1935364905949110532). 

So let's do a deep-dive and figure out what each part does. This is a high level breakdown of the cutting edge ideas on tinygrad's roadmap.

## Pytorch frontend:

![tinygrad-model-frontend](tinygrad_model.png)

From the outside, tinygrad feels like PyTorch. It has same  API and high-level functions. But under the hood it works totally differently. 

Yet at the same time, the tinygrad frontend is a bit cleaner and nicer than PyTorch's.

First, everything is a method on a tensor. In tinygrad you do
```py
y = Tensor.arange(10).relu()
```
but in PyTorch's verbose style you have to initialize a class for `relu` despite it being stateless
```py
x = torch.arange(10)
relu = torch.nn.ReLU()
y = relu(x)
```

Second, tinygrad is lazy. Nothing runs until you call `realize()`. This means the compiler sees your entire computation graph before executing anything, enabling insane operator fusion. In tinygrad you do
```py
Tensor.arange(10).realize()
```
but in pytorch you just do
```py
torch.arange(10)
```

Third, you do not need `nn.Module` or `forward() in tinygrad. You can just init a model
```py
class Model:
  def __init__(self): self.w1 = nn.Linear(4, 8)
  def __call__(self, x): return self.w1(x)
```
But in PyTorch you do
```py
class Model(nn.Module):
  def __init__(self): self.w1 = nn.Linear(4, 8)
  def forward(self, x): return self.w1(x)
```

After that most things are the same between tinygrad and PyTorch. The frontend in tinygrad is pretty much 100% complete. It is beautiful and simple.

## Halide rangeify

[Halide](https://github.com/halide/Halide) is a programming language that revolutionized image processing by separating *what* you compute from *how* you compute it.

Most languages lock them together. You write a for loop, and you've decided: it runs sequentially. You want parallelism? Rewrite the loop. Want to vectorize it? Rewrite it again. Want to use shared memory? Rewrite it again. The algorithm gets tangled with implementation details —- memory placement, thread hierarchy, synchronization.

Halide says: write the algorithm once. *Then* specify the **schedule**, how to execute it.

Tinygrad applies this principle to deep learning using an abstraction called **ranges**. Let's compute the query-key similarity score in attention to see why this matters. 

**The naive algorithm:**
```py
for q in range(num_queries):
  for k in range(num_keys):
    score[q, k] = query[q] @ key[k]     # read and write from global memory
```
This runs sequentially, reading and writing from global memory. It's correct but slow.

**Adding vectorization (UPCAST):**

```py
for q in range(0, num_queries, TILE_SIZE):              
  for k in range(0, num_keys, TILE_SIZE):     

    # Process TILE_SIZE×TILE_SIZE (q, k) pairs in parallel          
    score[q:q+TILE_SIZE, k:k+TILE_SIZE] = queries[q:q+TILE_SIZE] @ keys[k:k+TILE_SIZE]  # read and write from global memory
```
Now each operation processes a TILE_SIZE×TILE_SIZE block at once. The GPU tensor cores handle this as a single vectorized operation, not TILE_SIZE**2 individual operations. The algorithm is identical—we're just telling the compiler to batch iterations into vectors that are better suited for our hardware.

**Adding shared memory (LOCAL):**
```py
for q in range(0, num_queries, TILE_SIZE):  
  load_to_shared_mem(queries[q:q+TILE_SIZE])   # put in shared memory
  
  for k in range(0, num_keys, TILE_SIZE):             
    load_to_shared_mem(keys[k:k+TILE_SIZE])   # put in shared memory
    
    # Now read from fast shared memory instead of slow global memory
    score[q:q+TILE_SIZE, k:k+TILE_SIZE] = shared_queries[q:q+TILE_SIZE] @ shared_keys[k:k+TILE_SIZE]
```
The algorithm is still the same, but now:
* Queries stay in fast shared memory across all k iterations (reused many times)
* Keys are loaded once per k iteration
* All reads hit shared memory instead of global memory (huge speedup)

Wait a second, this is one of the innovations of Flash Attention: use tiling and shared memory. (Yes, this is still missing some of the core features of flash attention like not fully realizing the softmax matrix, but you get the point.) The algorithm didn't change, only the schedule

Like Halide, tinygrad does not hardcode one schedule. Instead, it exposes all possible schedules with axis types:

![ranges](rangeify.png)

Here is what each "axis" means

| Axis Type | Description |
|-----------|-------------|
| `GLOBAL` | Parallelizes loops across all global work items (GPU grid dimensions). |
| `LOCAL` | Parallelizes loops within a local workgroup (GPU threadblock). |
| `WARP` | Parallelizes loops within a warp/wavefront (GPU subgroup of threads). |
| `THREAD` | Parallelizes loops across individual threads (GPU thread). |
| `LOOP` | Creates a sequential loop that cannot be parallelized. |
| `UPCAST` | Vectorizes a loop by processing multiple elements per iteration. |
| `UNROLL` | Unrolls a loop by replicating its body multiple times to reduce branch overhead. |
| `GROUP_REDUCE` | Reduces values across a local workgroup using group operations. |
| `REDUCE` | Reduces values along a single axis (e.g., for accumulation). |

These axes allow you to manipulate for

In the previous code, we tagged the query and key axes with `UPCAST` and `LOCAL` tags to vectorize each for loop and put the data in shared memory.

In tinygrad, you can express all these different schedules with one algorithm. You're not locked into one way of running things. (tinygrad is [working](https://discord.com/channels/1068976834382925865/1255400554012741683/1428593384720699557) on adding another Axis Type called `Multi` which will parallelize loops across entire GPUs.)

PyTorch uses ShapeTrackers, which lock algorithm and schedule together. This severely limits the types of optimizations the PyTorch compiler can perform and is why tinygrad will win. [TVM](https://github.com/apache/tvm) is another deep learning library that is also powered by a Halide-style compiler.

[Halide paper](https://people.csail.mit.edu/jrk/halide-pldi13.pdf) | [Tinygrad's ranges explanation](https://x.com/__tinygrad__/status/1964037572503752910)


## E-Graph symbolic

![egglog](egglog.png)

E-graphs are data structures that simplify math equations. All the math we write in the pytorch frontend can be simplified into mathematically equivalent but cheaper/simpler equations with E-graphs.

As an example, consider `(x*2)//2`. It should simplify to `x`. But you could also rewrite `x*2` as `x<<1` first (bit shift is faster). Then you're stuck with `(x<<1)//2` and can't simplify further.

E-graphs figure out the optimal rewrite order and make sure you do not get stuck. Instead of just applying rules randomly, they capture the space of equivalent expressions and make it easy to find the cheapest one.

As far as I can tell, tinygrad does not currently use E-graphs. Instead, it uses [PatternMatcher](https://github.com/tinygrad/tinygrad/blob/457602b350fd6747b43c07ea54d142ac2a223f5e/tinygrad/uop/ops.py#L926) to simplify math equations. But it seems like they are aiming to switch to E-graphs in the future.

The best e-graph library at the moment is egglog ([paper](https://arxiv.org/abs/2304.04332), [code](https://egglog-python.readthedocs.io/latest/)). The deep learning compiler [Luminal](https://github.com/luminal-ai/luminal), backed by YC, uses egglog and the Carnigie Mellow project [mirage](https://github.com/mirage-project/mirage) uses egg, another e-graph. I'm not sure how PyTorch handles this...


## ILP Memory Planner (MODeL)

Meta published a paper, MODeL, showing you can reduce memory usage by 30% just by arranging tensors better in memory.

They key insight: deep learning has no branching. There are no if-statements, no surprise allocations, no dynamic control flow. The computation graph is static. This means you know in advance:
* Exactly which tensors you'll need
* Exactly when you'll need them
* Exactly when you can deallocate them

So the MODeL paper say we should treat this as a graph problem.
* Each node is an operation (mat-mul, convolution, relu, etc.)
* Each edge is a tensor (activations, model weights, optimizer states, gradient, etc.)

Each edge has a known size and lifetime. The edges that go into a node represent the tensors needed to perform that operation and the edge that comes out of that node represents the resultant computed tensor. The goal is to find a topological order (so the computation still runs in the correct order) that **minimizes peak memory usage**.

This is an integer linear programming (ILP) problem. Throw it at a solver and boom—30% savings on average.

This only works because DL is acyclic and deterministic. Traditional programming is Turing-complete—you can't know tensor lifetimes in advance. But DL is not—it's a restricted model that allows this kind of global optimization.

Right now tinygrad uses a much simpler memory planner, but they hope to implement this technique to build a deep learning compiler that is incredibly memory efficient.

Do any real-world DL libraries uses this? PyTorch? Jax? I'm not sure... Tinygrad has not implemented this yet either, but it is on their roadmap!

[MODeL paper](https://proceedings.mlr.press/v202/steiner23a/steiner23a.pdf)

## ThunderKittens backend

![thunderkittens](tk.png)

Chris Re's lab built [ThunderKittens](https://github.com/HazyResearch/ThunderKittens), a DSL for writing fast GPU kernels.

The key insight: NVIDIA GPUs have tensor cores. These are specialized hardware units that operate on **16×16 tiles of data**. Most programmers don't think in tiles - they think in individual threads. So kernel code is unnecessarily complicated.

ThunderKittens makes the tile the first-class citizen. Instead of managing thousands of individual thread operations, you manage 16x16 tiles. Working at a higher-level of abstraction, ThunderKittens greatly simplifies kernels code.

Remarkably, ThunderKittens surpass Flash Attention 2 performance in just a few lines of code. 

![tk-benchmark](attn-mini.png)

Tinygrad is building [TinyKittens](https://github.com/tinygrad/tinygrad/tree/master/extra/thunder) -- a similar abstraction for kernels. Instead of writing kernels at the lowest level, tinygrad will generate kernels at the higher level of abstraction, tiles. Tinygrad just started working on this.

PyTorch can generate kernels in cuda, which is very low level, or they just use tons of handwritten Cuda Kernels.

[ThunderKittens blog](https://hazyresearch.stanford.edu/blog/2024-05-12-quick-tk) | [ThunderKittens paper](https://arxiv.org/abs/2410.20399).

## Pure Python driver

![python-driver](tinygrad_vs_others.png)

Here's the insane part: tinygrad reverse-engineered CUDA and HIP (AMD's equivalent) and rewrote them in pure Python. Using `ctypes` from the standard Python library, they can interact with pointers and C-like code from within Python itself.

Why does this matter?

PyTorch is primarily locked into CUDA. If you want AMD, you need HIP. If you want Intel, you need something else. Each backend requires different libraries and code. Tinygrad says: we'll implement all of them in Python. One codebase, all backends. You can run on NVIDIA, AMD, Intel, Apple Meta -- all from the same Python code.

This is actually genius. You're not locked into CUDA's ecosystem anymore. You can run on AMD, Intel, Apple Metal—whatever—all from the same Python code.


## Why This Matters

Tinygrad isn’t copying PyTorch. It’s rebuilding deep learning from first principles — lazy tensors, symbolic rewrites, smart memory, tiled kernels — all in a few thousand lines of Python.

There is a surprising amount of recent research papers that are referenced here. And most surprisingly, how little of these techniques PyTorch uses.